
# MarketMind — SEC Filing Ingestion Pipeline

# Fetches 10-K, 10-Q, 8-K, Form 4 for top 50 US stocks → Qdrant Cloud


In [1]:
# Install dependencies
!pip install -q qdrant-client openai tiktoken tqdm requests lxml

In [2]:
# Imports
import os, re, time, uuid, hashlib, json, requests, xml.etree.ElementTree as ET
from lxml import etree as lxml_etree
from datetime import datetime, timedelta
from typing import Optional
from tqdm.notebook import tqdm
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct,
    Filter, FieldCondition, MatchValue, PayloadSchemaType
)

In [3]:
# Load Credentials
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
SEC_USER_AGENT = userdata.get("SEC_USER_AGENT")

_missing = [k for k, v in {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "QDRANT_URL":     QDRANT_URL,
    "QDRANT_API_KEY": QDRANT_API_KEY,
}.items() if not v]

if _missing:
    raise ValueError(
        f"Missing Colab secrets: {', '.join(_missing)}\n"
        f"Add them via the key icon in the left sidebar."
    )

print("All secrets loaded successfully")

All secrets loaded successfully


In [4]:
# Configuration

CHUNK_SIZE       = 500
CHUNK_OVERLAP    = 50
SEC_DELAY        = 0.15
EMBED_BATCH_SIZE = 100
EMBEDDING_MODEL  = "text-embedding-3-small"
EMBEDDING_DIMS   = 1536
COLLECTION_NAME  = "sec_filings"
MIN_TEXT_TOKENS  = 80
TOP_50 = [
    "AAPL","MSFT","NVDA","AMZN","GOOGL","META","TSLA","BRK-B",
    "AVGO","JPM","LLY","V","UNH","XOM","MA","COST","HD","PG",
    "JNJ","ABBV","BAC","KO","MRK","CVX","CRM","NFLX","AMD",
    "PEP","TMO","ORCL","ACN","MCD","ADBE","LIN","QCOM","TXN",
    "WMT","DIS","INTC","IBM","GE","CAT","BA","GS","MS",
    "BKNG","SPGI","ISRG","NOW","UBER"
]

FILING_CONFIG = {
    "10-K": {"count": 2,  "lookback_days": 800},
    "10-Q": {"count": 4,  "lookback_days": 400},
    "8-K":  {"count": 10, "lookback_days": 180},
    "4":    {"count": 10, "lookback_days": 90 },
}

# Section patterns for section-aware chunking
SECTION_PATTERNS = [
    ("Risk Factors",         r"item\s+1a[\.\s]"),
    ("Business Overview",    r"item\s+1[\.\s]"),
    ("MD&A",                 r"item\s+7[\.\s]"),
    ("Market Risk",          r"item\s+7a[\.\s]"),
    ("Financial Statements", r"item\s+8[\.\s]"),
    ("Legal Proceedings",    r"item\s+3[\.\s]"),
    ("Properties",           r"item\s+2[\.\s]"),
    ("Quantitative Risk",    r"item\s+7a[\.\s]"),
    ("Controls",             r"item\s+9a[\.\s]"),
]


In [5]:
# Clients

openai_client = OpenAI(api_key=OPENAI_API_KEY)
qdrant        = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
SEC_HEADERS   = {"User-Agent": SEC_USER_AGENT, "Accept-Encoding": "gzip, deflate"}
print("Clients initialized")

Clients initialized


In [6]:
# Qdrant collection setup

def init_collection():
    existing = [c.name for c in qdrant.get_collections().collections]
    if COLLECTION_NAME not in existing:
        qdrant.create_collection(
            COLLECTION_NAME,
            vectors_config=VectorParams(size=EMBEDDING_DIMS, distance=Distance.COSINE)
        )
        print(f"Created: {COLLECTION_NAME}")
        for field, schema in [
            ("ticker",       PayloadSchemaType.KEYWORD),
            ("filing_type",  PayloadSchemaType.KEYWORD),
            ("filing_date",  PayloadSchemaType.KEYWORD),
            ("section",      PayloadSchemaType.KEYWORD),
            ("content_hash", PayloadSchemaType.KEYWORD),
            ("filed_at",     PayloadSchemaType.INTEGER),
            ("doc_type",     PayloadSchemaType.KEYWORD),  # "text" or "structured"
        ]:
            try:
                qdrant.create_payload_index(COLLECTION_NAME, field, schema)
            except Exception:
                pass
    else:
        count = qdrant.count(COLLECTION_NAME).count
        print(f"Collection exists — {count:,} vectors")

init_collection()

Collection exists — 1,604 vectors


In [7]:
# SEC request with retry

def sec_get(url: str, timeout: int = 30, max_retries: int = 5) -> requests.Response:
    """
    GET with automatic retry and exponential backoff.
    Handles 429 Too Many Requests and transient 5xx errors.
    """
    for attempt in range(max_retries):
        try:
            time.sleep(SEC_DELAY)
            r = requests.get(url, headers=SEC_HEADERS, timeout=timeout)

            if r.status_code == 429:
                wait = 2 ** attempt * 5   # 5, 10, 20, 40, 80 seconds
                print(f"  429 rate limit — waiting {wait}s (attempt {attempt+1})")
                time.sleep(wait)
                continue

            if r.status_code in (500, 502, 503):
                wait = 2 ** attempt * 2
                print(f"  {r.status_code} server error — waiting {wait}s")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r

        except requests.exceptions.Timeout:
            if attempt == max_retries - 1:
                raise
            print(f"  Timeout — retrying ({attempt+1}/{max_retries})")
            time.sleep(2 ** attempt)

        except requests.exceptions.ConnectionError:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

    raise RuntimeError(f"Failed after {max_retries} attempts: {url}")




In [8]:
# CIK lookup

_cik_cache = {}

def get_cik(ticker: str) -> Optional[str]:
    if ticker in _cik_cache:
        return _cik_cache[ticker]
    r = sec_get("https://www.sec.gov/files/company_tickers.json")
    for entry in r.json().values():
        t = entry.get("ticker", "").upper()
        _cik_cache[t] = str(entry["cik_str"]).zfill(10)
    return _cik_cache.get(ticker.upper())


In [9]:
# handle >1000 filings with pagination

def get_filings_list(cik: str, form_type: str,
                     count: int, lookback_days: int) -> list[dict]:
    """
    Fetch filings list. Handles companies with >1000 filings by
    following the older-filings archive links in the submissions JSON.
    """
    cutoff  = datetime.now() - timedelta(days=lookback_days)
    results = []

    def _extract(recent: dict):
        forms    = recent.get("form", [])
        dates    = recent.get("filingDate", [])
        accnos   = recent.get("accessionNumber", [])
        primdocs = recent.get("primaryDocument", [])

        for form, date_str, acc, doc in zip(forms, dates, accnos, primdocs):
            if form != form_type:
                continue
            try:
                filing_date = datetime.strptime(date_str, "%Y-%m-%d")
            except Exception:
                continue
            if filing_date < cutoff:
                return False   # gone past lookback — stop pagination
            results.append({
                "form_type":   form,
                "filing_date": date_str,
                "accession":   acc.replace("-", ""),
                "primary_doc": doc,
                "cik":         cik,
            })
            if len(results) >= count:
                return False   # have enough
        return True   # need to paginate

    # Fetch main submissions JSON
    r    = sec_get(f"https://data.sec.gov/submissions/CIK{cik}.json")
    data = r.json()
    need_more = _extract(data["filings"]["recent"])

    # FIX 1: paginate through older filings if needed
    if need_more:
        for older_file in data["filings"].get("files", []):
            if not need_more:
                break
            r2   = sec_get(f"https://data.sec.gov/{older_file['name']}")
            need_more = _extract(r2.json())

    return results


In [10]:
# Robust URL construction

def build_filing_url(cik: str, accession: str, primary_doc: str) -> str:
    """
    Build filing URL handling subdirectory paths in primary_doc.
    primary_doc can be:
      - "aapl-20240928.htm"              → simple
      - "d123456/aapl-20240928.htm"      → subdirectory
    """
    cik_int = int(cik)
    # primary_doc sometimes includes a subdirectory — use it as-is
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik_int}/{accession}/{primary_doc}"
    )

In [11]:
# Document type detection and text extraction

def extract_text(content: str, url: str) -> tuple[str, str]:
    """
    Returns (clean_text, doc_format) where doc_format is html/txt/xml.
    Handles HTML, plain text, and XML filing formats correctly.
    """
    url_lower = url.lower()

    # Detect format by URL extension
    if url_lower.endswith(".xml") or url_lower.endswith(".xsd"):
        return _extract_xml_narrative(content), "xml"

    if url_lower.endswith(".txt"):
        # Plain text — minimal cleaning needed
        text = re.sub(r"\s+", " ", content).strip()
        return text, "txt"

    # Default: HTML — strip tags
    text = re.sub(r"<[^>]+>", " ", content)           # remove tags
    text = re.sub(r"&nbsp;",  " ", text)               # decode common entities
    text = re.sub(r"&amp;",   "&", text)
    text = re.sub(r"&lt;",    "<", text)
    text = re.sub(r"&gt;",    ">", text)
    text = re.sub(r"&#\d+;",  " ", text)               # numeric entities
    text = re.sub(r"\s+",     " ", text).strip()

    # Remove lines that are pure numbers/whitespace (XBRL inline data)
    lines = [l.strip() for l in text.split(".")
             if len(l.strip()) > 40 and not re.match(r"^[\d\s\$\,\.\-\%]+$", l.strip())]
    return ". ".join(lines), "html"


def _extract_xml_narrative(content: str) -> str:
    """Extract any readable text nodes from XML, skipping tags."""
    try:
        root = ET.fromstring(content)
        texts = []
        for elem in root.iter():
            if elem.text and len(elem.text.strip()) > 20:
                texts.append(elem.text.strip())
        return " ".join(texts)
    except ET.ParseError:
        # Fallback: strip angle brackets
        return re.sub(r"<[^>]+>", " ", content)

In [12]:
# Form 4 parsed as structured data, not embedded

def parse_form4(content: str, ticker: str, filing: dict) -> list[dict]:
    """
    Parse Form 4 XML using exact schema structure confirmed from debug:
    - All field values are wrapped in a <value> child tag
    - transactionCode lives inside <transactionCoding> not <transactionAmounts>
    - transactionPricePerShare may have only <footnoteId> with no <value>
      (undisclosed price — treat as None, do not skip the transaction)
    """
    transactions = []

    try:
        parser        = lxml_etree.XMLParser(recover=True, encoding="utf-8")
        content_bytes = content.encode("utf-8") if isinstance(content, str) else content
        root          = lxml_etree.fromstring(content_bytes, parser=parser)

        def val(parent, tag: str) -> str | None:
            """
            Fetch text from <tag><value>TEXT</value></tag> pattern.
            Falls back to direct text if no <value> child exists.
            Returns None if tag not found or has no text.
            """
            el = parent.find(f".//{tag}")
            if el is None:
                return None
            # Try nested <value> first (most fields)
            v = el.find("value")
            text = (v.text if v is not None else el.text)
            return text.strip() if text and text.strip() else None

        # Document-level fields
        issuer_name = val(root, "issuerName")  or ticker
        owner_name  = val(root, "rptOwnerName") or "Unknown Insider"
        owner_title = val(root, "officerTitle") or "Insider"

        # Both non-derivative and derivative transactions
        all_txns = (
            root.findall(".//nonDerivativeTransaction") +
            root.findall(".//derivativeTransaction")
        )

        if not all_txns:
            print(f"    Form 4: XML parsed but 0 transactions found")
            return []

        code_map = {
            "P": "purchased",
            "S": "sold",
            "A": "was awarded",
            "M": "exercised options for",
            "G": "gifted",
            "F": "shares withheld (tax)",
            "D": "disposed of",
            "J": "other transaction in",
        }

        for txn in all_txns:
            # transactionCode is inside transactionCoding
            txn_code = val(txn, "transactionCode") or "U"
            txn_date = val(txn, "transactionDate")   # <transactionDate><value>
            shares   = val(txn, "transactionShares") # <transactionShares><value>
            price    = val(txn, "transactionPricePerShare")  # may be None

            # Skip if missing the two required fields
            if not shares or not txn_date:
                continue

            try:
                shares_f   = float(shares)
                action     = code_map.get(txn_code, f"transacted ({txn_code}) in")
                price_str  = f"at ${float(price):.2f} per share" if price else \
                             "at an undisclosed price"

                sentence = (
                    f"{owner_name} ({owner_title}) {action} "
                    f"{shares_f:,.0f} shares of {issuer_name} "
                    f"{price_str} on {txn_date}."
                )

                chash = hashlib.sha256(
                    f"{ticker}{txn_date}{shares}{price or ''}"
                    f"{filing['accession']}{txn_code}".encode()
                ).hexdigest()

                transactions.append({
                    "ticker":        ticker,
                    "filing_type":   "4",
                    "filing_date":   filing["filing_date"],
                    "filed_at":      int(datetime.strptime(
                                         filing["filing_date"], "%Y-%m-%d"
                                     ).timestamp()),
                    "section":       "Insider Transactions",
                    "text":          sentence,
                    "doc_type":      "structured",
                    "insider_name":  owner_name,
                    "insider_title": owner_title,
                    "action":        action,
                    "txn_code":      txn_code,
                    "shares":        shares_f,
                    "price":         float(price) if price else None,
                    "txn_date":      txn_date,
                    "content_hash":  chash,
                    "accession":     filing["accession"],
                })

            except (ValueError, TypeError):
                continue

    except Exception as e:
        print(f"    Form 4 parse error: {e}")

    return transactions


In [13]:
# Section-aware chunking with context carry-forward

def detect_section(text: str) -> Optional[str]:
    """Returns section name if this chunk contains a section header."""
    text_lower = text.lower()
    for name, pattern in SECTION_PATTERNS:
        if re.search(pattern, text_lower):
            return name
    return None


def chunk_with_sections(text: str) -> list[tuple[str, str]]:
    """
    FIX 4: Carry section context forward across chunks.
    Returns list of (chunk_text, section_name) tuples.
    Chunks deep inside a section inherit the section from the header chunk.
    """
    import tiktoken
    enc    = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)

    chunks  = []
    start   = 0
    current_section = "General"   # default until first header found

    while start < len(tokens):
        end        = min(start + CHUNK_SIZE, len(tokens))
        chunk_text = enc.decode(tokens[start:end])

        # Check if this chunk contains a new section header
        detected = detect_section(chunk_text)
        if detected:
            current_section = detected   # update running section context

        chunks.append((chunk_text, current_section))
        start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks

In [14]:
# Robust text quality check

def is_quality_text(text: str) -> bool:
    """
    FIX 3: Token-based quality check (stricter than char count).
    Rejects: empty text, pure numbers/tables, XBRL data dumps.
    """
    if not text or len(text.strip()) < 100:
        return False

    import tiktoken
    enc    = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)

    if len(tokens) < MIN_TEXT_TOKENS:
        return False

    # Reject if >70% of content is numeric/punctuation (financial tables)
    alpha_chars = sum(1 for c in text if c.isalpha())
    if alpha_chars / max(len(text), 1) < 0.30:
        return False

    return True

In [15]:
# Dedup check

def chunk_exists(content_hash: str) -> bool:
    results, _ = qdrant.scroll(
        COLLECTION_NAME,
        scroll_filter=Filter(must=[
            FieldCondition(key="content_hash",
                           match=MatchValue(value=content_hash))
        ]),
        limit=1
    )
    return len(results) > 0


def content_hash(text: str, ticker: str, filing_date: str) -> str:
    return hashlib.sha256(
        f"{ticker}{filing_date}{text[:200]}".encode()
    ).hexdigest()

In [16]:
# Embedding with retry

def embed_batch(texts: list[str]) -> list[list[float]]:
    for attempt in range(3):
        try:
            r = openai_client.embeddings.create(
                model=EMBEDDING_MODEL, input=texts)
            return [item.embedding for item in r.data]
        except Exception as e:
            if attempt == 2:
                raise
            print(f"    embed retry {attempt+1}: {e}")
            time.sleep(2 ** attempt)

In [17]:
# Store chunks

def store_text_chunks(chunks_with_sections: list[tuple[str, str]],
                      ticker: str, filing: dict) -> int:
    stored = 0
    new_texts, new_hashes, new_sections = [], [], []

    for chunk_text, section in chunks_with_sections:
        if not is_quality_text(chunk_text):      # FIX 3
            continue
        chash = content_hash(chunk_text, ticker, filing["filing_date"])
        if not chunk_exists(chash):
            new_texts.append(chunk_text)
            new_hashes.append(chash)
            new_sections.append(section)

    if not new_texts:
        return 0

    for i in range(0, len(new_texts), EMBED_BATCH_SIZE):
        batch_texts    = new_texts[i:i + EMBED_BATCH_SIZE]
        batch_hashes   = new_hashes[i:i + EMBED_BATCH_SIZE]
        batch_sections = new_sections[i:i + EMBED_BATCH_SIZE]
        vectors        = embed_batch(batch_texts)

        points = []
        for text, vector, chash, section in zip(
                batch_texts, vectors, batch_hashes, batch_sections):
            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload={
                    "ticker":        ticker,
                    "filing_type":   filing["form_type"],
                    "filing_date":   filing["filing_date"],
                    "filed_at":      int(datetime.strptime(
                                         filing["filing_date"], "%Y-%m-%d"
                                     ).timestamp()),
                    "section":       section,
                    "text":          text,
                    "doc_type":      "text",
                    "content_hash":  chash,
                    "accession":     filing["accession"],
                }
            ))

        qdrant.upsert(COLLECTION_NAME, points=points)
        stored += len(points)
        time.sleep(0.05)

    return stored


def store_structured_points(records: list[dict]) -> int:
    """Store Form 4 structured records — embed the sentence text."""
    new_records = [r for r in records if not chunk_exists(r["content_hash"])]
    if not new_records:
        return 0

    texts   = [r["text"] for r in new_records]
    vectors = embed_batch(texts)

    points = [
        PointStruct(id=str(uuid.uuid4()), vector=vec, payload=rec)
        for rec, vec in zip(new_records, vectors)
    ]
    qdrant.upsert(COLLECTION_NAME, points=points)
    return len(points)


In [18]:
# Checkpoint helpers

CHECKPOINT_FILE = "/content/sec_checkpoint.json"

def load_checkpoint() -> dict:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {}

def save_checkpoint(cp: dict) -> None:
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(cp, f, indent=2)


In [24]:
# Main pipeline
def run_pipeline(tickers: list[str] = TOP_50, resume: bool = True):
    checkpoint = load_checkpoint() if resume else {}
    stats = {"filings": 0, "chunks": 0, "form4_txns": 0,
             "skipped": 0, "errors": 0}

    print(f"\nStarting pipeline — {len(tickers)} tickers\n")

    for ticker in tqdm(tickers, desc="Tickers"):
        print(f"\n{'─'*50}\n{ticker}")

        cik = get_cik(ticker)
        if not cik:
            print(f"  CIK not found — skipping")
            stats["errors"] += 1
            continue

        for form_type, cfg in FILING_CONFIG.items():
            try:
                filings = get_filings_list(        # FIX 1: paginated
                    cik, form_type,
                    cfg["count"], cfg["lookback_days"]
                )
            except Exception as e:
                print(f"  {form_type} list error: {e}")
                stats["errors"] += 1
                continue

            if not filings:
                print(f"  {form_type}: no filings in lookback window")
                continue

            print(f"  {form_type}: {len(filings)} filing(s)")

            for filing in filings:
                # FIX 2: use accession in key — prevents date collision
                # (multiple Form 4s filed same day by different insiders)
                filing_key = f"{ticker}_{form_type}_{filing['accession']}"

                if filing_key in checkpoint:
                    print(f"    {filing['filing_date']}: already done, skipping")
                    stats["skipped"] += 1
                    continue

                url = build_filing_url(
                    cik, filing["accession"], filing["primary_doc"])

                try:
                    r = sec_get(url, timeout=45)
                except Exception as e:
                    print(f"    {filing['filing_date']}: fetch failed — {e}")
                    stats["errors"] += 1
                    continue

                if form_type == "4":
                    # primary_doc is like "xslF345X06/form4.xml" or
                    # "xslF345X06/wk-form4_1234567.xml" — strip the
                    # stylesheet subdirectory and use the actual filename
                    cik_int      = int(cik)
                    xml_filename = filing["primary_doc"].split("/")[-1]
                    raw_xml_url  = (
                        f"https://www.sec.gov/Archives/edgar/data/"
                        f"{cik_int}/{filing['accession']}/{xml_filename}"
                    )
                    try:
                        xml_r   = sec_get(raw_xml_url, timeout=30)
                        content = xml_r.text
                    except Exception:
                        # last fallback — try standard form4.xml name
                        try:
                            fallback_url = (
                                f"https://www.sec.gov/Archives/edgar/data/"
                                f"{cik_int}/{filing['accession']}/form4.xml"
                            )
                            xml_r   = sec_get(fallback_url, timeout=30)
                            content = xml_r.text
                        except Exception as e:
                            print(f"    Form 4 fetch failed: {e}")
                            stats["errors"] += 1
                            continue

                    records = parse_form4(content, ticker, filing)
                    if records:
                        stored = store_structured_points(records)
                        stats["form4_txns"] += stored
                        print(f"    {filing['filing_date']}: "
                              f"{stored} insider transactions stored")
                        # FIX 3: only checkpoint on success
                        checkpoint[filing_key] = {
                            "processed_at": datetime.now().isoformat(),
                            "stored": stored
                        }
                        save_checkpoint(checkpoint)
                    else:
                        # FIX 3: don't checkpoint failed parses
                        print(f"    {filing['filing_date']}: "
                              f"no transactions parsed — will retry next run")

                else:
                    text, fmt = extract_text(r.text, url)
                    print(f"    {filing['filing_date']}: "
                          f"format={fmt} raw_chars={len(text):,}")

                    if not is_quality_text(text):
                        print(f"    {filing['filing_date']}: "
                              f"failed quality check — skipping")
                        continue

                    chunks_with_sections = chunk_with_sections(text)
                    print(f"    {filing['filing_date']}: "
                          f"{len(chunks_with_sections)} chunks")

                    stored = store_text_chunks(
                        chunks_with_sections, ticker, filing)
                    stats["chunks"]  += stored
                    stats["filings"] += 1
                    print(f"    stored {stored} new chunks")

                    # Only checkpoint text filings on success
                    if stored > 0:
                        checkpoint[filing_key] = {
                            "processed_at": datetime.now().isoformat(),
                            "stored": stored
                        }
                        save_checkpoint(checkpoint)

    print(f"\n{'═'*50}")
    print(f"Filings processed    : {stats['filings']}")
    print(f"Text chunks stored   : {stats['chunks']:,}")
    print(f"Form 4 txns stored   : {stats['form4_txns']:,}")
    print(f"Skipped (cached)     : {stats['skipped']}")
    print(f"Errors               : {stats['errors']}")
    print(f"Qdrant total vectors : {qdrant.count(COLLECTION_NAME).count:,}")



In [21]:
# Run

# Test with 3 first
#run_pipeline(tickers=["AAPL", "MSFT", "NVDA"], resume=True)


Starting pipeline — 3 tickers



Tickers:   0%|          | 0/3 [00:00<?, ?it/s]


──────────────────────────────────────────────────
AAPL
  10-K: 2 filing(s)
    2025-10-31: format=html raw_chars=212,072
    2025-10-31: 101 chunks
    stored 0 new chunks
    2024-11-01: format=html raw_chars=210,182
    2024-11-01: 101 chunks
    stored 0 new chunks
  10-Q: 3 filing(s)
    2026-01-30: format=html raw_chars=64,085
    2026-01-30: 37 chunks
    stored 0 new chunks
    2025-08-01: format=html raw_chars=69,240
    2025-08-01: 42 chunks
    stored 0 new chunks
    2025-05-02: format=html raw_chars=80,781
    2025-05-02: 46 chunks
    stored 0 new chunks
  8-K: 5 filing(s)
    2026-02-24: format=html raw_chars=5,224
    2026-02-24: 4 chunks
    stored 0 new chunks
    2026-01-29: format=html raw_chars=3,680
    2026-01-29: 3 chunks
    stored 0 new chunks
    2026-01-02: format=html raw_chars=3,986
    2026-01-02: 3 chunks
    stored 0 new chunks
    2025-12-05: format=html raw_chars=3,680
    2025-12-05: 3 chunks
    stored 0 new chunks
    2025-10-30: format=html raw_c

In [26]:
# Full run
run_pipeline(tickers=TOP_50, resume=True)


Starting pipeline — 50 tickers



Tickers:   0%|          | 0/50 [00:00<?, ?it/s]


──────────────────────────────────────────────────
AAPL
  10-K: 2 filing(s)
    2025-10-31: format=html raw_chars=212,072
    2025-10-31: 101 chunks
    stored 0 new chunks
    2024-11-01: format=html raw_chars=210,182
    2024-11-01: 101 chunks
    stored 0 new chunks
  10-Q: 3 filing(s)
    2026-01-30: format=html raw_chars=64,085
    2026-01-30: 37 chunks
    stored 0 new chunks
    2025-08-01: format=html raw_chars=69,240
    2025-08-01: 42 chunks
    stored 0 new chunks
    2025-05-02: format=html raw_chars=80,781
    2025-05-02: 46 chunks
    stored 0 new chunks
  8-K: 5 filing(s)
    2026-02-24: format=html raw_chars=5,224
    2026-02-24: 4 chunks
    stored 0 new chunks
    2026-01-29: format=html raw_chars=3,680
    2026-01-29: 3 chunks
    stored 0 new chunks
    2026-01-02: format=html raw_chars=3,986
    2026-01-02: 3 chunks
    stored 0 new chunks
    2025-12-05: format=html raw_chars=3,680
    2025-12-05: 3 chunks
    stored 0 new chunks
    2025-10-30: format=html raw_c

In [40]:
import pkg_resources
print(pkg_resources.get_distribution("qdrant-client").version)

1.17.1


In [45]:
# Verify
def verify():
    total = qdrant.count(COLLECTION_NAME).count
    print(f"Total vectors: {total:,}\n")

    def search(query_text, filters, limit=3):
        vec = embed_batch([query_text])[0]
        return qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=vec,
            query_filter=Filter(must=filters),
            limit=limit,
            with_payload=True
        ).points

    print("Sample — AAPL Risk Factors:")
    results = search(
        "What are the main business risks for Apple?",
        [
            FieldCondition(key="ticker",      match=MatchValue(value="AAPL")),
            FieldCondition(key="filing_type", match=MatchValue(value="10-K")),
            FieldCondition(key="section",     match=MatchValue(value="Risk Factors")),
        ]
    )
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] score={r.score:.3f} | {r.payload.get('section')} | {r.payload.get('filing_date')}")
        print("  " + r.payload["text"][:300] + "...")

    print("\nVector counts per filing type (AAPL):")
    for ft in ["10-K", "10-Q", "8-K", "4"]:
        count = qdrant.count(
            COLLECTION_NAME,
            count_filter=Filter(must=[
                FieldCondition(key="ticker",      match=MatchValue(value="AAPL")),
                FieldCondition(key="filing_type", match=MatchValue(value=ft)),
            ])
        ).count
        print(f"  {ft:5s}: {count:>5,} vectors")

    print("\nSample — AAPL insider transactions:")
    results2 = search(
        "Apple executive sold shares",
        [
            FieldCondition(key="ticker",      match=MatchValue(value="AAPL")),
            FieldCondition(key="filing_type", match=MatchValue(value="4")),
        ]
    )
    for r in results2:
        print(f"  {r.payload['text']}")

    print("\nVectors per ticker (first 10):")
    for ticker in TOP_50[:10]:
        count = qdrant.count(
            COLLECTION_NAME,
            count_filter=Filter(must=[
                FieldCondition(key="ticker", match=MatchValue(value=ticker))
            ])
        ).count
        print(f"  {ticker:8s}: {count:>5,}")

verify()

Total vectors: 36,025

Sample — AAPL Risk Factors:

[1] score=0.569 | Risk Factors | 2024-11-01
   be identified by words such as future, anticipates, believes, estimates, expects, intends, plans, predicts, will, would, could, can, may, and similar terms. Forward-looking statements are not guarantees of future performance and the Company s actual results may differ significantly from the results...

[2] score=0.555 | Risk Factors | 2025-10-31
  53 Part IV Item 15. Exhibit and Financial Statement Schedules 54 Item 16. Form 10-K Summary 57 This Annual Report on Form 10-K ( Form 10-K ) contains forward-looking statements, within the meaning of the Private Securities Litigation Reform Act of 1995, that involve risks and uncertainties. Many of ...

[3] score=0.546 | Risk Factors | 2025-10-31
   website URLs are intended to be inactive textual references only. Risk Factors The following summarizes factors that could have a material adverse effect on the Company s business, reputation, result